# CheckSent-BN Sentiment Classification
**Model:** meta-llama/Llama-3.1-8B-Instruct  
**Strategy:** CoT-style

## 1. Import Libraries

In [1]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import gc
import re
import random

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("Libraries imported successfully.")
print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

Libraries imported successfully.
PyTorch: 2.11.0+cu128
Transformers: 5.16.1
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


## 2. Configuration

In [3]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

DATA_DIR = (
    "/kaggle/input/datasets/"
    "taufiqzahantushar/checksent-bn/"
    "CheckSent-BN"
)

TRAIN_PATH = os.path.join(
    DATA_DIR,
    "train_data.tsv"
)

TEST_PATH = os.path.join(
    DATA_DIR,
    "test_data.tsv"
)

MAX_NEW_TOKENS = 16

LABELS = [
    "NEGATIVE",
    "NEUTRAL",
    "POSITIVE"
]

## 3. Load Dataset

In [4]:
train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t"
)

test_df = pd.read_csv(
    TEST_PATH,
    sep="\t"
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTest sentiment distribution:")
print(test_df["Sentiment"].value_counts())

Train shape: (9254, 6)
Test shape: (2314, 6)

Test sentiment distribution:
Sentiment
NEGATIVE    1123
NEUTRAL      825
POSITIVE     366
Name: count, dtype: int64


## 4. Data Validation

In [5]:
required_columns = [
    "ID",
    "Date",
    "News Portal",
    "Headline",
    "Claim",
    "Sentiment"
]

assert all(
    column in train_df.columns
    for column in required_columns
)

assert all(
    column in test_df.columns
    for column in required_columns
)

assert train_df["Headline"].isna().sum() == 0
assert test_df["Headline"].isna().sum() == 0

assert train_df["Sentiment"].isna().sum() == 0
assert test_df["Sentiment"].isna().sum() == 0

assert set(test_df["Sentiment"].unique()) == set(LABELS)

print("\nDataset validation passed.")


Dataset validation passed.


## 5. Hugging Face Token

In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("gptoss")

print("Hugging Face token loaded.")

Hugging Face token loaded.


## 6. Tokenizer

In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully.


## 7. 4-bit Quantization

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit quantization configured.")

4-bit quantization configured.


## 8. Load Model

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("Model loaded successfully.")
print("Model device:", model.device)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded successfully.
Model device: cuda:0


## 9. Sentiment Parser

In [10]:
def parse_sentiment(output):

    if output is None:
        return None

    output = output.strip().upper()

    output = output.replace("`", "")
    output = output.replace("*", "")
    output = output.replace("\n", " ")

    if output == "NEGATIVE":
        return "NEGATIVE"

    if output == "NEUTRAL":
        return "NEUTRAL"

    if output == "POSITIVE":
        return "POSITIVE"

    matches = []

    for label in LABELS:

        if re.search(
            rf"\b{label}\b",
            output
        ):
            matches.append(label)

    if len(matches) == 1:
        return matches[0]

    return None

## 10. CoT System Prompt

In [11]:
SYSTEM_PROMPT = """
You are performing Bengali sentiment classification.

Classify the sentiment of a Bengali news headline into exactly
ONE of these labels:

NEGATIVE:
The headline expresses negative sentiment, negative events,
criticism, conflict, loss, danger, failure, or other negative
emotional polarity.

NEUTRAL:
The headline is factual, informational, descriptive, or does
not clearly express positive or negative sentiment.

POSITIVE:
The headline expresses positive sentiment, success,
achievement, benefit, celebration, praise, or other positive
emotional polarity.

Before making the classification, carefully analyze:

1. The overall meaning of the headline.
2. The emotional polarity expressed.
3. Whether the headline is mainly factual or evaluative.
4. Positive and negative sentiment-bearing expressions.
5. The overall sentiment rather than isolated words.

Do not provide your reasoning.

Return ONLY ONE label.

Allowed labels:
NEGATIVE
NEUTRAL
POSITIVE
""".strip()

## 11. Single Sample Prediction

In [12]:
def predict_sentiment(headline):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Headline:\n{headline}\n\n"
                "Final label:"
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[
        0,
        input_length:
    ]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    prediction = parse_sentiment(
        generated_text
    )

    return prediction, generated_text

## 12. Sample Test

In [13]:
sample_indices = [
    0,
    1,
    2,
    10,
    20
]

print("Sample predictions:\n")

for idx in sample_indices:

    headline = test_df.iloc[idx]["Headline"]
    actual = test_df.iloc[idx]["Sentiment"]

    prediction, raw_output = predict_sentiment(
        headline
    )

    print("=" * 80)
    print("Index:", idx)
    print("Headline:", headline)
    print("Actual:", actual)
    print("Prediction:", prediction)
    print("Raw output:", repr(raw_output))

Sample predictions:



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Index: 0
Headline: বেঙ্গালুরু-দমদম বিমানে বোমাতঙ্ক! সোশাল মিডিয়ায় লাগাতার হুমকি পোস্টে আতঙ্ক
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 1
Headline: নেপালে নদীতে পড়ল ভারতীয় পর্যটকবোঝাই বাস, মৃত অন্তত ১৪
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 2
Headline: পুজো হবে নির্বিঘ্নে, আশ্বাস ইউনুস সরকারের, মহালয়ার প্রস্তুতি বাংলাদেশে
Actual: NEUTRAL
Prediction: POSITIVE
Raw output: 'POSITIVE'
Index: 10
Headline: প্রবল বর্ষণে বিপর্যস্ত মেঘালয়, হড়পা বান ও ধসে মৃত ১০ জন
Actual: NEGATIVE
Prediction: NEGATIVE
Raw output: 'NEGATIVE'
Index: 20
Headline: ইউটিউবে রোনাল্ডোর সাতকাহন, কোন সাত কারণে সিআর সেভেনের চ্যানেলে বুঁদ ভক্তরা?
Actual: POSITIVE
Prediction: POSITIVE
Raw output: 'POSITIVE'


## 13. Full CoT Evaluation

In [14]:
cot_predictions = []
cot_raw_outputs = []

print("\nStarting CoT evaluation...\n")

for idx, headline in enumerate(
    tqdm(
        test_df["Headline"],
        total=len(test_df),
        desc="CoT"
    )
):

    try:

        prediction, raw_output = predict_sentiment(
            headline
        )

    except Exception as e:

        print(
            f"\nError at index {idx}: {e}"
        )

        prediction = None
        raw_output = ""

    cot_predictions.append(
        prediction
    )

    cot_raw_outputs.append(
        raw_output
    )

print("\nCoT inference completed.")

print(
    "Total predictions:",
    len(cot_predictions)
)

print(
    "Unknown predictions:",
    sum(
        p is None
        for p in cot_predictions
    )
)


Starting CoT evaluation...



CoT:   0%|          | 0/2314 [00:00<?, ?it/s]


CoT inference completed.
Total predictions: 2314
Unknown predictions: 0


## 14. Prepare Predictions

In [15]:
y_true = test_df["Sentiment"].tolist()

y_pred = [
    prediction
    if prediction in LABELS
    else "UNKNOWN"
    for prediction in cot_predictions
]

print("\nPrediction distribution:")
print(
    pd.Series(y_pred).value_counts()
)


Prediction distribution:
NEGATIVE    1129
POSITIVE     927
NEUTRAL      258
Name: count, dtype: int64


## 15. Calculate Metrics

In [16]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision, recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABELS,
        average="macro",
        zero_division=0
    )
)

## 16. Main Results

In [17]:
print("\n" + "=" * 60)
print("LLAMA 3.1 8B INSTRUCT — COT RESULTS")
print("=" * 60)

print(
    f"Accuracy : {accuracy:.4f} "
    f"({accuracy * 100:.2f}%)"
)

print(
    f"Precision: {precision:.4f} "
    f"({precision * 100:.2f}%)"
)

print(
    f"Recall   : {recall:.4f} "
    f"({recall * 100:.2f}%)"
)

print(
    f"Macro F1 : {macro_f1:.4f} "
    f"({macro_f1 * 100:.2f}%)"
)

print(
    f"Unknown  : {sum(p is None for p in cot_predictions)}"
)

print(
    f"Total test samples: {len(y_true)}"
)


LLAMA 3.1 8B INSTRUCT — COT RESULTS
Accuracy : 0.6595 (65.95%)
Precision: 0.6886 (68.86%)
Recall   : 0.6843 (68.43%)
Macro F1 : 0.5955 (59.55%)
Unknown  : 0
Total test samples: 2314


## 17. Class-wise Results

In [18]:
print("\n" + "=" * 60)
print("CLASS-WISE RESULTS")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        labels=LABELS,
        target_names=LABELS,
        digits=4,
        zero_division=0
    )
)


CLASS-WISE RESULTS
              precision    recall  f1-score   support

    NEGATIVE     0.8592    0.8638    0.8615      1123
     NEUTRAL     0.8411    0.2630    0.4007       825
    POSITIVE     0.3657    0.9262    0.5244       366

    accuracy                         0.6595      2314
   macro avg     0.6886    0.6843    0.5955      2314
weighted avg     0.7747    0.6595    0.6439      2314



## 18. Confusion Matrix

In [19]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
)

print("\nConfusion Matrix:")
display(cm_df)


Confusion Matrix:


,NEGATIVE,NEUTRAL,POSITIVE
NEGATIVE,970,31,122
NEUTRAL,142,217,466
POSITIVE,17,10,339


## 19. Final Result Table

In [20]:
cot_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Macro F1"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        macro_f1
    ]
})

print("\nFinal CoT Results:")
display(cot_results)


Final CoT Results:


,Metric,Score
0,Accuracy,0.659464
1,Precision,0.688649
2,Recall,0.684339
3,Macro F1,0.595519
